# 1-1. 루프 제어

선택 심화. `1`을 마친 뒤 본다.

`1`의 루프는 모델이 도구를 부를지, 몇 개를 부를지, 언제 멈출지를 전부 모델에 맡겼다.
여기서는 그 결정에 손잡이 넷을 단다. `tool_choice`, `parallel_tool_calls`, 추론 effort, 추론 아이템 되돌리기.
같은 과제를 손잡이만 바꿔 돌리고 정답률, 턴, 호출, 비용을 잰다.

### 공통 설정

`1`과 같은 블록이다. 실행은 하위 모델로 한다. 손잡이의 효과가 하위 모델에서 더 크게 드러난다.

In [1]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp


def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total

도구 셋과 오류 처리는 `1`의 것 그대로다.

In [10]:
CORPUS = Path("../../../../../data/corpus/graph")


def list_docs():
    return [f.stem for f in sorted(CORPUS.glob("*.md"))]


def grep_docs(pattern):
    hits = []
    for f in sorted(CORPUS.glob("*.md")):
        n = f.read_text(encoding="utf-8").count(pattern)
        if n:
            hits.append({"title": f.stem, "count": n})
    return hits


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(title)
    return f.read_text(encoding="utf-8")[:2000]


TOOLS = [
    {"type": "function", "name": "list_docs", "strict": True,
     "description": "코퍼스에 있는 문서 제목을 모두 돌려준다.",
     "parameters": {"type": "object", "properties": {}, "required": [],
                    "additionalProperties": False}},
    {"type": "function", "name": "grep_docs", "strict": True,
     "description": "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.",
     "parameters": {"type": "object",
                    "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                    "required": ["pattern"], "additionalProperties": False}},
    {"type": "function", "name": "read_doc", "strict": True,
     "description": "문서 한 편의 본문 앞부분을 돌려준다.",
     "parameters": {"type": "object",
                    "properties": {"title": {"type": "string",
                                             "description": "문서 제목. list_docs 가 준 것 그대로"}},
                    "required": ["title"], "additionalProperties": False}},
]
REGISTRY = {"list_docs": list_docs, "grep_docs": grep_docs, "read_doc": read_doc}


def call_tool(name, args):
    try:
        return json.dumps(REGISTRY[name](**args), ensure_ascii=False)
    except FileNotFoundError as e:
        return f"error: '{e}' 라는 제목의 문서가 없다. list_docs 로 제목을 확인하고 그대로 다시 부른다."
    except Exception as e:
        return f"error: {type(e).__name__} {e}"

### 과제와 채점

`1`의 과제를 여섯 쌍으로 늘린다. 두 낱말을 모두 언급한 문서 수를 묻는다.
`grep_docs`를 두 번 부르고 두 목록의 교집합을 세야 한다. 정답은 같은 일을 코드로 해서 만든다.

In [11]:
PAIRS = [("한국독립당", "조선민족혁명당"), ("광복군", "임시의정원"), ("의열단", "김원봉"),
         ("하와이", "대한인국민회"), ("신민회", "안창호"), ("상하이", "블라디보스토크")]


def both(a, b):
    return len({d["title"] for d in grep_docs(a)} & {d["title"] for d in grep_docs(b)})


QS = [(f"이 문서 묶음에서 '{a}'와 '{b}'를 모두 언급한 문서는 몇 편인가. 마지막 줄에 숫자만 쓴다.", both(a, b))
      for a, b in PAIRS]
print([answer for _, answer in QS])

[6, 5, 8, 6, 5, 10]


마지막 줄의 마지막 숫자를 답으로 본다.

In [ ]:
def correct(text, answer):
    lines = (text or "").strip().splitlines()
    nums = re.findall(r"\d+", lines[-1]) if lines else []
    return bool(nums) and int(nums[-1]) == answer

### 손잡이가 달린 루프

`1`의 루프에 인자 넷을 더한다. `choice`는 턴 번호를 받아 그 턴의 `tool_choice`를 돌려주는 함수다. 턴마다 다르게 걸 수 있어야 뒤의 실험이 된다.
결과는 답, 턴, 도구 호출 수, 시간, 정지 사유를 담은 딕셔너리로 돌려준다.

In [ ]:
def run(task, model=LOWER, max_turns=8, choice=lambda turn: "auto", parallel=True,
        effort=None, keep_reasoning=True):  # auto : 호출할 때 알아서 호출하라 / parallel : 병렬 호출 여부 / effort : reasoning effort(low, mid, high, extra..)
    messages = [{"role": "user", "content": task}]
    n_calls, first, t0 = 0, None, time.perf_counter()
    for turn in range(1, max_turns + 1):
        kw = dict(model=model, input=messages, tools=TOOLS, store=False,
                  tool_choice=choice(turn), parallel_tool_calls=parallel)   # llm에게 전달할 arguments / 멀티턴 조정 시 previous id보다 직접 주는게 좋음
        if effort:
            kw["reasoning"] = {"effort": effort}
            if keep_reasoning:  # reasoning한 결과를 포함할 것인가?
                kw["include"] = ["reasoning.encrypted_content"]
        r = keep(client.responses.create(**kw))
        calls = [i for i in r.output if i.type == "function_call"]
        first = first or [c.name for c in calls]
        if not calls:
            return {"text": r.output_text, "turns": turn, "calls": n_calls, "first": first,
                    "sec": time.perf_counter() - t0, "stop": "완료"}
        n_calls += len(calls)
        messages += [i.model_dump(exclude_none=True) for i in r.output
                     if keep_reasoning or i.type != "reasoning"]
        for c in calls:
            messages.append({"type": "function_call_output", "call_id": c.call_id,
                             "output": call_tool(c.name, json.loads(c.arguments))})
    return {"text": None, "turns": max_turns, "calls": n_calls, "first": first,
            "sec": time.perf_counter() - t0, "stop": "반복 예산"}

여섯 문항을 동시에 돌리고 한 줄로 요약하는 함수다. 결과는 `RESULTS`에 모아 끝에서 나란히 둔다.

In [ ]:
RESULTS = {}


# qs 의 질문을 6개씩 동시에 run 으로 돌려 정답 수, 턴, 호출, 비용을 label 이름으로 RESULTS 에 남긴다. kw(model, max_turns, choice, parallel, effort, keep_reasoning)는 run 에 그대로 넘긴다
def batch(label, qs=QS, **kw):
    mark = len(USAGE)
    with ThreadPoolExecutor(6) as ex:
        outs = list(ex.map(lambda q: run(q[0], **kw), qs))
    row = {"정답": sum(correct(o["text"], a) for o, (_, a) in zip(outs, qs)),
           "도구 안 씀": sum(o["calls"] == 0 for o in outs),
           "답 없음": sum(o["text"] is None for o in outs),
           "턴": sum(o["turns"] for o in outs) / len(outs),
           "호출": sum(o["calls"] for o in outs),
           "초": sum(o["sec"] for o in outs) / len(outs),
           "비용": dollars(USAGE[mark:])}
    RESULTS[label] = row
    print(f"{label:16s} 정답 {row['정답']}/{len(qs)}  도구 안 씀 {row['도구 안 씀']}  답 없음 {row['답 없음']}"
          f"  턴 {row['턴']:.1f}  호출 {row['호출']}  {row['초']:.1f}초  ${row['비용']:.4f}")
    return outs

손잡이를 하나도 건드리지 않은 기준선이다.

In [ ]:
base = batch("기준선")

### 1. tool_choice: 도구를 부를지 누가 정하는가

기본값 `auto`는 모델이 정한다. `required`는 반드시 도구를 하나 이상 부르게 하고, `none`은 도구를 못 부르게 한다.
특정 도구 이름을 주면 그 도구를 부르게 한다.

외운 내용으로 답할 수 있는 질문이라면 모델이 도구를 건너뛸 수도 있다. 위키백과로 답할 수 있는 질문 넷으로 확인한다.

In [ ]:
GENERAL = [("강우규가 1919년에 저격하려 한 인물은 누구인가", None),
           ("김구의 호는 무엇인가", None),
           ("이봉창 의거는 몇 년에 일어났나", None),
           ("안중근이 이토 히로부미를 사살한 도시는 어디인가", None)]

for label, choice in [("auto", lambda t: "auto"), ("첫 턴 required", lambda t: "required" if t == 1 else "auto")]:
    outs = [run(q, choice=choice) for q, _ in GENERAL]
    print(f"{label:14s} 도구를 안 쓴 답 {sum(o['calls'] == 0 for o in outs)}/{len(outs)}  호출 {sum(o['calls'] for o in outs)}")

이 모델은 `auto`에서도 대개 도구를 부른다. 그래도 그것은 모델의 선택이다. 근거가 있어야 하는 답이면 첫 턴에 `required`를 걸어 선택지를 없앤다.
첫 턴에만 거는 이유는 다음 셀에서 본다. 계속 걸어 두면 모델은 답을 쓸 수 없다. 답을 쓰는 턴은 도구를 부르지 않는 턴이기 때문이다.

In [ ]:
outs = batch("계속 required", choice=lambda t: "required", max_turns=4)
print([o["stop"] for o in outs])

특정 도구를 지정하면 첫 호출을 고정할 수 있다. 제목을 정확히 알아야 하는 과제에서 `list_docs`를 먼저 부르게 하는 식이다. (force)

In [ ]:
force = {"type": "function", "name": "list_docs"}
o = run("'김구 (독립운동가)' 문서의 첫머리를 읽는다", choice=lambda t: force if t == 1 else "auto")
print("첫 턴에 부른 도구", o["first"], "  전체 호출", o["calls"], "  ", o["stop"])

`none`은 예산이 끝날 때 쓴다. `1`의 `run_budgeted`는 예산이 끝나면 답 없이 멈췄다.
마지막 턴에 `none`을 걸면 그때까지 모은 결과로 답을 쓰게 할 수 있다. 반복 예산을 두 턴으로 좁혀 비교한다.

In [ ]:
batch("2턴 예산", max_turns=2)
batch("2턴 + 마지막 none", max_turns=2, choice=lambda t: "none" if t == 2 else "auto")

`none`이 붙은 쪽은 답이 없는 경우가 사라진다. 그 답이 맞는지는 별개다. 덜 모은 상태에서 쓴 답이므로 정지 사유와 함께 남겨야 한다(`1`의 `why`).

### 2. parallel_tool_calls: 한 턴에 몇 개를 부르는가

기본값은 한 턴에 여러 호출을 허용한다. 끄면 한 턴에 하나씩 부른다.

In [ ]:
batch("병렬 끔", parallel=False)

끄면 같은 호출을 여러 턴에 나눠 하므로 턴이 는다. 정답률은 이 손잡이로 바뀌지 않는다.
그래도 끄는 경우가 있다. **앞 호출의 결과를 보고 다음 호출의 인자를 정해야 하거나, 도구가 부작용이 있어 순서가 중요할 때**다.

### 3. 추론(Reasoning) effort: 호출 사이에 생각하게 하는가

기준선은 effort 를 지정하지 않았다. 이 모델은 그때 추론 없이 바로 도구를 부르고 답을 쓴다.
effort 를 주면 호출 사이에 추론 아이템이 생긴다. 단계별로 돌린다.

In [ ]:
for e in ["low", "medium"]:
    batch(f"effort {e}", effort=e)

기준선의 틀린 답 하나를 열어 본다. 무엇을 부르고 무엇을 답했는지 정답과 대조한다.

In [ ]:
miss = next(((o, q) for o, q in zip(base, QS) if not correct(o["text"], q[1])), None)
if miss:
    o, (q, answer) = miss
    print(q)
    print("정답", answer, " 호출", o["calls"], " 턴", o["turns"])
    print(o["text"][-300:])

도구 결과는 받았는데 마지막 계산에서 틀리는 경우가 많다. 두 목록에서 겹치는 제목을 세는 일을 추론 없이 한 번에 쓰다 보니 생기는 실수다.

상위 모델에서도 같은 일이 생기는지 effort 없이 돌려 본다. (tool call 문제가 아니라, reasoning이나 generation에서 문제가 발생)

In [ ]:
batch("상위 모델", model=MODEL)

### 4. 추론 아이템을 되돌리는가

`store=False`로 입력 배열을 직접 관리하면 추론 아이템도 직접 되돌려야 한다.
서버에 남지 않으므로 `include=["reasoning.encrypted_content"]`로 암호화된 추론 내용을 받아 다음 요청에 그대로 넣는다.
빼고 보내도 요청은 성공한다. 모델이 직전 턴에 무엇을 생각했는지를 모른 채 이어 갈 뿐이다.

In [ ]:
batch("medium 추론 버림", effort="medium", keep_reasoning=False)

두세 턴에 끝나는 이 과제에서는 버린 쪽의 손해가 드러나지 않는다. 턴이 긴 과제에서는 따로 재야 한다.

### 나란히 둔다

In [ ]:
for label, r in RESULTS.items():
    print(f"{label:18s} 정답 {r['정답']}/{len(QS)}  답 없음 {r['답 없음']}  턴 {r['턴']:.1f}"
          f"  호출 {r['호출']:3d}  {r['초']:4.1f}초  ${r['비용']:.4f}")

### 정리

정답률을 가장 크게 바꾼 손잡이는 effort 다. 추론 없이 돈 하위 모델은 필요한 도구를 다 부르고도 마지막 계산에서 자주 틀렸고, 호출도 많았다.
effort 를 주면 정답률이 오르고 호출이 줄어 비용도 함께 내려간다. 상위 모델로 바꾸는 것보다 하위 모델에 effort 를 주는 쪽이 정답률이 높고 쌌다. (ReAct)

`tool_choice`는 정답률보다 루프의 모양을 정한다. `required`는 근거가 꼭 필요한 첫 턴에만 걸고, `none`은 예산이 끝나는 마지막 턴에 건다. `required`를 계속 걸면 루프가 끝나지 않는다.
`parallel_tool_calls`를 끄면 턴만 늘어난다. 순서가 중요한 도구가 아니면 켜 둔다.

### 비용 확인

In [ ]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")